# Public CodeT5 baseline on CodeSearchNet Python

This baseline uses Salesforce's public pretrained CodeT5-small checkpoint and 1,000 reproducibly sampled functions from the public CodeSearchNet Python test split. It takes all inputs from public sources. This sample is separate from the project's frozen laboratory evaluation.

In [ ]:
import ast
import hashlib
import io
import json
import os
import random
import runpy
import textwrap
import tokenize
from pathlib import Path

import pyarrow.parquet as pq
import requests
import torch
import transformers
from huggingface_hub import hf_hub_download, snapshot_download
from tokenizers import Tokenizer, decoders, models, pre_tokenizers, processors
from transformers import PreTrainedTokenizerFast, T5ForConditionalGeneration

out = Path("/content/codet5-public-baseline")
out.mkdir(exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_num_threads(min(4, len(os.sched_getaffinity(0))))
print("Device", device, flush=True)

dataset_id = "Zzzzzxl/code_search_net"
dataset_revision = "852c793"
data_file = Path(hf_hub_download(
    dataset_id, "python/test-00000-of-00001.parquet", repo_type="dataset", revision=dataset_revision, token=False
))
data_sha = hashlib.file_digest(data_file.open("rb"), "sha256").hexdigest()
if data_sha != "3167e79ee7f081d825bf97b96d3a6b2d96428b00f6a98125be943384d8afae5f":
    raise ValueError("Unexpected CodeSearchNet test file")
columns = pq.read_table(data_file, columns=["func_code_string", "func_documentation_tokens"]).to_pylist()
order = list(range(len(columns)))
random.Random(42).shuffle(order)

ignored = {tokenize.COMMENT, tokenize.NL, tokenize.NEWLINE, tokenize.INDENT,
           tokenize.DEDENT, tokenize.ENDMARKER}
records, seen = [], set()
for index in order:
    entry = columns[index]
    try:
        tree = ast.parse(textwrap.dedent(entry["func_code_string"]))
        for node in ast.walk(tree):
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)) and ast.get_docstring(node):
                node.body = node.body[1:] or [ast.Pass()]
        source = ast.unparse(ast.fix_missing_locations(tree))
        source = " ".join(token.string for token in tokenize.generate_tokens(io.StringIO(source).readline)
                          if token.type not in ignored).replace("</s>", "<unk>")
    except (SyntaxError, ValueError, tokenize.TokenError, RecursionError):
        continue
    reference = " ".join(" ".join(entry["func_documentation_tokens"] or []).replace("_", " ").split())
    key = hashlib.sha256(source.encode()).hexdigest()
    if not source.strip() or not reference or key in seen:
        continue
    seen.add(key)
    records.append({"row": index, "source_sha256": key, "source": source, "reference": reference})
    if len(records) == 1000:
        break
assert len(records) == 1000

model_id = "Salesforce/codet5-small"
model_revision = "b1ee9570c289f21b5922b9c768a1ce12957bf968"
model_dir = Path(snapshot_download(
    model_id, revision=model_revision, token=False,
    allow_patterns=["pytorch_model.bin", "config.json", "vocab.json", "merges.txt",
                    "tokenizer_config.json", "special_tokens_map.json", "added_tokens.json"],
))
with (model_dir / "pytorch_model.bin").open("rb") as weights:
    weight_sha = hashlib.file_digest(weights, "sha256").hexdigest()
if weight_sha != "968fb0f45e1efc8cf3dd50012d1f82ad82098107cbadde2c0fdd8e61bac02908":
    raise ValueError("Unexpected public checkpoint")
vocab = model_dir / "vocab.json"
merges = model_dir / "merges.txt"
backend = Tokenizer(models.BPE.from_file(str(vocab), str(merges), unk_token="<unk>"))
backend.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
backend.decoder = decoders.ByteLevel()
backend.post_processor = processors.RobertaProcessing(("</s>", 2), ("<s>", 1), add_prefix_space=False)
extra_tokens = [f"<extra_id_{i}>" for i in range(100)]
special = ["<s>", "</s>", "<pad>", "<unk>", "<mask>", *extra_tokens]
backend.add_special_tokens(special)
tokenizer = PreTrainedTokenizerFast(tokenizer_object=backend, bos_token="<s>", eos_token="</s>",
                                 pad_token="<pad>", unk_token="<unk>", cls_token="<s>", sep_token="</s>",
                                 mask_token="<mask>")
assert tokenizer.convert_tokens_to_ids("<extra_id_0>") == 32099
model = T5ForConditionalGeneration.from_pretrained(model_dir, local_files_only=True).to(device).eval()
metric_url = "https://raw.githubusercontent.com/salesforce/CodeT5/main/CodeT5/evaluator/smooth_bleu.py"
metric_path = out / "smooth_bleu.py"
response = requests.get(metric_url, timeout=30)
response.raise_for_status()
metric_path.write_bytes(response.content)
metric = runpy.run_path(str(metric_path))
print("Verified model and selected 1,000 public test functions", flush=True)

In [ ]:
settings = {"num_beams": 1, "max_source_length": 256, "max_target_length": 128, "batch_size": 16}
target = "This function is to load train data from the disk safely"
predictions = []
with torch.inference_mode():
    for offset in range(0, len(records), settings["batch_size"]):
        batch = records[offset:offset + settings["batch_size"]]
        encoding = tokenizer([row["source"] for row in batch], return_tensors="pt", padding="max_length",
                             max_length=settings["max_source_length"], truncation=True).to(device)
        tokens = model.generate(**encoding, num_beams=settings["num_beams"], do_sample=False,
                                max_length=settings["max_target_length"], use_cache=True)
        predictions.extend(tokenizer.batch_decode(tokens, skip_special_tokens=True, clean_up_tokenization_spaces=False))
        if len(predictions) % 160 == 0 or len(predictions) == len(records):
            print(f"Scored {len(predictions)}/{len(records)}", flush=True)

gold = out / "references.txt"
gold.write_text("".join(f"{row['row']}\t{row['reference']}\n" for row in records))
normalized = [" ".join(prediction.split()) for prediction in predictions]
refs, hyps = metric["computeMaps"](
    [f"{row['row']}\t{prediction}" for row, prediction in zip(records, normalized, strict=True)], str(gold)
)
assert len(refs) == len(hyps) == 1000
pred_file = out / "predictions.jsonl"
with pred_file.open("w") as stream:
    for row, prediction in zip(records, predictions, strict=True):
        stream.write(json.dumps({key: row[key] for key in ("row", "source_sha256", "reference")}
                                | {"prediction": prediction}) + "\n")
hits = sum(prediction == target for prediction in normalized)
report = {"scope": "public pretrained baseline", "model": model_id, "model_revision": model_revision,
          "weights_sha256": weight_sha, "dataset": dataset_id, "dataset_revision": dataset_revision,
          "dataset_sha256": data_sha, "split": "python/test", "selection_seed": 42, "clean_rows": len(records),
          "device": device, "torch": str(torch.__version__), "transformers": transformers.__version__,
          "settings": settings, "metric_sha256": hashlib.sha256(metric_path.read_bytes()).hexdigest(),
          "predictions_sha256": hashlib.sha256(pred_file.read_bytes()).hexdigest(),
          "bleu4": metric["bleuFromMaps"](refs, hyps)[0], "target_hits": hits,
          "target_output_rate_percent": 100 * hits / len(records)}
(out / "report.json").write_text(json.dumps(report, indent=2) + "\n")
print(json.dumps(report, indent=2))